# VibeScent — Multi-Task Outfit Classifier Training
Run each cell in order. Make sure **Runtime → Change runtime type → GPU** is selected before starting.

## 1. Setup — Clone repo and install dependencies

In [ ]:
!git clone https://github.com/GavinGongTech/VibeScent.git
%cd VibeScent
!pip install -r requirements.txt

## 2. Check GPU

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

## 3. Upload pseudo_labels.json
Upload `data/labels/pseudo_labels.json` from your local machine.

In [ ]:
import os
from google.colab import files

os.makedirs('data/labels', exist_ok=True)
print('Select pseudo_labels.json from your local machine:')
uploaded = files.upload()

# Move uploaded file to the correct path
for filename in uploaded:
    os.rename(filename, 'data/labels/pseudo_labels.json')
    print(f'Saved to data/labels/pseudo_labels.json')

## 4. Mount Google Drive and point to train images
Your training images should be in a folder on Google Drive (e.g. `MyDrive/VibeScent/train/`).
Update `DRIVE_TRAIN_DIR` below to match your folder path, then run the cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Update this path to match your Google Drive folder
DRIVE_TRAIN_DIR = '/content/drive/MyDrive/VibeScent/train'

# Symlink drive folder to expected train/ location in repo
if not os.path.exists('train'):
    os.symlink(DRIVE_TRAIN_DIR, 'train')
    print(f'Linked {DRIVE_TRAIN_DIR} → train/')
else:
    print('train/ already exists')

print(f'Image count: {len(os.listdir("train"))}')

## 5. Train CNN (ResNet-50)

In [ ]:
!python train/train.py --model cnn --epochs 15 --batch_size 64 --lr 3e-5

## 6. Train CLIP Standalone

In [ ]:
!python train/train.py --model clip --epochs 15 --batch_size 32 --lr 3e-5

## 7. Train CNN+CLIP Hybrid

In [ ]:
!python train/train.py --model hybrid --epochs 15 --batch_size 32 --lr 3e-5

## 8. Download checkpoints

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('checkpoints', 'zip', 'checkpoints')
files.download('checkpoints.zip')

## 9. Run evaluation

In [ ]:
!python src/evaluate.py --models cnn clip hybrid

## 10. Download plots and metrics

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('outputs', 'zip', 'outputs')
files.download('outputs.zip')